# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [ ]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V2.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

Dim_Klant (SCD TYPE 2) ROBBERT

In [ ]:
# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met deze logica:
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude actuele rij afsluiten + nieuwe INSERT
# - bestaande klant, niet gewijzigd -> niets doen.

# business key = klantnr uit SDM
# surrogate key = klant_sk in DWH

# We halen eerst brondata op uit beide klanttabellen.
# Dit doen we omdat Dim_Klant niet uit één bron komt, maar uit de tabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
# We voegen deze dus daarom ook verticaal samen.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# We willen deze twee losse DataFrames verticaal samenvoegen.
# We maken dus één bronset voor Dim_Klant.
df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

# We verwijderen dubbele klanten op basis van de business key klantnr.
df_klant_source = df_klant_source.drop_duplicates(subset=["klantnr"]).reset_index(drop=True)

# Nu halen we de actuele klantrecords op uit het DWH.
# Bij SCD Type 2 willen we namelijk alleen vergelijken met de actuele rij van een klant,
# niet met de oude historische versies.
# We herkennen de actuele regel aan: eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

# Nu gaan we bepalen of een bestaande klant veranderd is.
# We vergelijken alleen inhoudelijke klantgegevens.
# We vergelijken dus NIET op:
# - klant_sk
# - begintijd
# - eindtijd
# want die horen bij de DWH-historie en niet bij de business-inhoud van de klant.

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# Eerst bepalen hoeveel klanten nieuw, gewijzigd of ongewijzigd zijn.
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]
        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

# We gaan nu de echte ETL uitvoeren voor Dim_Klant
now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    # 1. Nieuwe klant -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1

    # 2. Bestaande klant -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Oude actuele rij afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Nieuwe actuele versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
                """, (
                int(src_row["klantnr"]),
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1
        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}"
)

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, klant_sk
""", dwh_conn)

print(result_df)

Dim_Accessoire (SCD TYPE 1) ROBBERT

Dim_Datum ROBBERT

In [ ]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0
logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))
        inserted_count += 1

dwh_conn.commit()
logger.info(f"Dim_Datum klaar | inserted={inserted_count}")

# 7. Controle

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

Fct_Verkoop ROBBERT


In [ ]:
# Wat willen we bereiken?
# We willen Fct_Verkoop vullen met verkoopgebeurtenissen uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop

# In het DWH komt dit samen in één feittabel:
# Fct_Verkoop
# De feittabel bevat:
# - verkoopnr
# - verkoop_type
# - datum_sk
# - aantal
# - totaalprijs
# - standaardprijs
# - klant_sk
# - fiets_sk of accessoire_sk
# - monteur_sk

# 1. Brondata ophalen uit beide verkoopbronnen
# We voegen nu ook verkoop_type toe, zodat de samengestelde PK
# (verkoopnr, verkoop_type) correct gevuld kan worden.

query_fiets_verkoop = """
SELECT
    fv.fiets_verkoopnr AS verkoopnr,
    'fiets' AS verkoop_type,
    fv.datum,
    fv.aantal,
    fv.verkoopprijs AS totaalprijs,
    ff.standaardprijs,
    fv.klant,
    fv.fiets,
    fv.monteur,
    NULL AS accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff
    ON fv.fiets = ff.fietsnr
"""

query_accessoire_verkoop = """
SELECT
    av.accessoire_verkoopnr AS verkoopnr,
    'accessoire' AS verkoop_type,
    av.datum,
    av.aantal,
    av.verkoopprijs AS totaalprijs,
    aa.standaardprijs,
    av.klant,
    NULL AS fiets,
    av.monteur,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa
    ON av.accessoire = aa.accessoirenr
"""

df_fiets_verkoop = pd.read_sql_query(query_fiets_verkoop, sdm_conn)
df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)

logger.info(f"Aantal verkooprijen uit Fiets_Verkoop: {len(df_fiets_verkoop)}")
logger.info(f"Aantal verkooprijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop)}")

# 2. Verticale samenvoeging tot één bronset
# Er is geen overlap-probleem meer, omdat verkoopnr nu samen
# met verkoop_type de sleutel vormt.

df_verkoop_source = pd.concat(
    [df_fiets_verkoop, df_accessoire_verkoop],
    ignore_index=True
)

logger.info(f"Totaal aantal verkooprijen in bronset: {len(df_verkoop_source)}")

# 3. Actuele dimensies uit het DWH ophalen
# Feiten moeten verwijzen naar de actuele surrogate keys van de dimensies.
df_datum_dwh = pd.read_sql_query("""
SELECT
    datum_sk,
    datum
FROM Dim_Datum
""", dwh_conn)

df_klant_dwh = pd.read_sql_query("""
SELECT
    klant_sk,
    klantnr
FROM Dim_Klant
WHERE eindtijd IS NULL
""", dwh_conn)

df_fiets_dwh = pd.read_sql_query("""
SELECT
    fiets_sk,
    fietsnr
FROM Dim_Fiets
WHERE eindtijd IS NULL
""", dwh_conn)

df_accessoire_dwh = pd.read_sql_query("""
SELECT
    accessoire_sk,
    accessoirenr
FROM Dim_Accessoire
WHERE eindtijd IS NULL
""", dwh_conn)

df_monteur_dwh = pd.read_sql_query("""
SELECT
    monteur_sk,
    monteurnr
FROM Dim_Monteur
WHERE eindtijd IS NULL
""", dwh_conn)

# 4. Bestaande verkooprecords uit DWH ophalen
# Nu op basis van de samengestelde sleutel.
df_verkoop_dwh = pd.read_sql_query("""
SELECT
    verkoopnr,
    verkoop_type
FROM Fct_Verkoop
""", dwh_conn)

logger.info(f"Aantal bestaande verkooprijen in DWH: {len(df_verkoop_dwh)}")

# 5. Helperfuncties voor surrogate key lookups
def get_datum_sk(datum_value):
    match = df_datum_dwh[df_datum_dwh["datum"] == str(datum_value)]
    if match.empty:
        raise ValueError(f"Geen datum_sk gevonden voor datum: {datum_value}")
    return int(match.iloc[0]["datum_sk"])

def get_klant_sk(klantnr):
    match = df_klant_dwh[df_klant_dwh["klantnr"] == int(klantnr)]
    if match.empty:
        raise ValueError(f"Geen klant_sk gevonden voor klantnr: {klantnr}")
    return int(match.iloc[0]["klant_sk"]) 

def get_fiets_sk(fietsnr):
    match = df_fiets_dwh[df_fiets_dwh["fietsnr"] == int(fietsnr)]
    if match.empty:
        raise ValueError(f"Geen fiets_sk gevonden voor fietsnr: {fietsnr}")
    return int(match.iloc[0]["fiets_sk"])

def get_accessoire_sk(accessoirenr):
    match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == int(accessoirenr)]
    if match.empty:
        raise ValueError(f"Geen accessoire_sk gevonden voor accessoirenr: {accessoirenr}")
    return int(match.iloc[0]["accessoire_sk"])

def get_monteur_sk(monteurnr):
    match = df_monteur_dwh[df_monteur_dwh["monteurnr"] == int(monteurnr)]
    if match.empty:
        raise ValueError(f"Geen monteur_sk gevonden voor monteurnr: {monteurnr}")
    return int(match.iloc[0]["monteur_sk"])

# 6. Vooraf bepalen hoeveel verkooprijen nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_verkoop_source.iterrows():
    verkoopnr = int(src_row["verkoopnr"])
    verkoop_type = src_row["verkoop_type"]

    current_match = df_verkoop_dwh[
        (df_verkoop_dwh["verkoopnr"] == verkoopnr) &
        (df_verkoop_dwh["verkoop_type"] == verkoop_type)
    ]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe verkooprijen: {new_count}")
logger.info(f"Aantal bestaande verkooprijen: {existing_count}")

# 7. Echte ETL uitvoeren voor Fct_Verkoop
# Omdat dit een feittabel is:
# - nieuwe verkoop -> INSERT
# - bestaande verkoop -> niets doen

inserted_count = 0
skipped_count = 0

logger.info("Start ETL voor Fct_Verkoop")

for _, src_row in df_verkoop_source.iterrows():
    verkoopnr = int(src_row["verkoopnr"])
    verkoop_type = src_row["verkoop_type"]

    current_match = df_verkoop_dwh[
        (df_verkoop_dwh["verkoopnr"] == verkoopnr) &
        (df_verkoop_dwh["verkoop_type"] == verkoop_type)
    ]

    # Alleen nieuwe feiten invoegen
    if current_match.empty:
        datum_sk = get_datum_sk(src_row["datum"])
        klant_sk = get_klant_sk(src_row["klant"])
        monteur_sk = get_monteur_sk(src_row["monteur"])

        # Business rule:
        # of fiets, of accessoire
        if pd.notna(src_row["fiets"]):
            fiets_sk = get_fiets_sk(src_row["fiets"])
            accessoire_sk = None
        else:
            fiets_sk = None
            accessoire_sk = get_accessoire_sk(src_row["accessoire"])

        dwh_conn.execute("""
            INSERT INTO Fct_Verkoop (
                verkoopnr,
                verkoop_type,
                datum_sk,
                aantal,
                totaalprijs,
                standaardprijs,
                klant_sk,
                fiets_sk,
                accessoire_sk,
                monteur_sk
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            verkoopnr,
            verkoop_type,
            datum_sk,
            int(src_row["aantal"]),
            float(src_row["totaalprijs"]),
            float(src_row["standaardprijs"]),
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk
        ))
        inserted_count += 1
    else:
        skipped_count += 1

dwh_conn.commit()
logger.info(
    f"Fct_Verkoop klaar | inserted={inserted_count}, skipped_existing={skipped_count}"
)

# 8. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Fct_Verkoop
    ORDER BY verkoopnr, verkoop_type
""", dwh_conn)

print(result_df)

Dim_Fiets (SCD Type 2) MEES

In [ ]:
logger.info("Ophalen Fiets uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, naam, adres, plaats
FROM (
    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Inkoop_Fiets f
    JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Verkoop_Fiets f
    JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Onderhoud_Fiets f
    JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
)
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
    fiets_sk
FROM Dim_Fiets
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

for row in source_data:
    (fietsnr, soort, merk, type_, kleur,
     fabrikantnr, fab_naam, fab_adres, fab_plaats) = row

    now = datetime.now()

    if fietsnr not in dwh_data:
        # nieuw
        logger.info(f"Nieuwe fiets: {fietsnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (fietsnr, soort, merk, type_, kleur,
              fabrikantnr, fab_naam, fab_adres, fab_plaats,
              now))

    else:
        # check wijzigingen
        dwh_row = dwh_data[fietsnr]

        (_, d_soort, d_merk, d_type, d_kleur,
         d_fabnr, d_fabnaam, d_fabadres, d_fabplaats,
         fiets_sk) = dwh_row

        if (soort, merk, type_, kleur,
            fabrikantnr, fab_naam, fab_adres, fab_plaats) != \
           (d_soort, d_merk, d_type, d_kleur,
            d_fabnr, d_fabnaam, d_fabadres, d_fabplaats):

            logger.info(f"Update in fiets: {fietsnr}")

            # 1. Sluit oude (voeg eindtijd toe)
            dwh_cursor.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, fiets_sk))

            # 2. Maak nieuwe (met nieuwe gegevens en begintijd)
            dwh_cursor.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (fietsnr, soort, merk, type_, kleur,
                  fabrikantnr, fab_naam, fab_adres, fab_plaats,
                  now))

dwh_conn.commit()
logger.info("Fiets dimensie geupdate in DWH.")

Dim_Monteur (SCD Type 1) MEES

In [ ]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr, naam, woonplaats, uurloon,
    filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (monteurnr, naam, woonplaats, uurloon,
     filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()

    if monteurnr not in dwh_data:
        # nieuwe regels worden toegevoegd
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr, naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (monteurnr, naam, woonplaats, uurloon,
              filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
              now))

    else:
        # bestaande regels worden overschreven als er wijzigingen zijn
        (_, d_naam, d_woonplaats, d_uurloon,
         d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie,
         monteur_sk) = dwh_data[monteurnr]

        if (naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) != \
           (d_naam, d_woonplaats, d_uurloon,
            d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?, woonplaats = ?, uurloon = ?,
                filiaalnr = ?, filiaalnaam = ?, filiaaladres = ?, filiaalprovincie = ?
            WHERE monteur_sk = ?
            """, (naam, woonplaats, uurloon,
                  filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
                  monteur_sk))
            
dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

Fct_Onderhoud MEES

In [ ]:
logger.info("Ophalen Onderhoud uit SDM")

sdm_cursor.execute("""
SELECT 
    o.onderhoudnr,
    o.datum,
    o.starttijd,
    o.eindtijd,
    o.fiets,
    o.monteur
FROM Onderhoud o
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT onderhoudnr FROM Fct_Onderhoud")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_or_create_datum_sk(datum):
    dwh_cursor.execute("SELECT datum_sk FROM Dim_Datum WHERE datum = ?", (datum,))
    if r := dwh_cursor.fetchone():
        return r[0]

    from datetime import datetime
    d = datetime.strptime(datum, "%Y-%m-%d")

    dwh_cursor.execute("""
    INSERT INTO Dim_Datum (datum, year, month, day)
    VALUES (?, ?, ?, ?)
    """, (datum, d.year, d.month, d.day))

    return dwh_cursor.lastrowid


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None


logger.info("Start ETL Fct_Onderhoud")

for row in source_data:
    onderhoudnr, datum, starttijd, eindtijd, fietsnr, monteurnr = row

    # skip bestaande
    if onderhoudnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    monteur_sk = get_monteur_sk(monteurnr)

    # Check of alle keys bestaan
    if not all([datum_sk, fiets_sk, monteur_sk]):
        logger.warning(f"SK ontbreekt voor onderhoud {onderhoudnr}")
        continue

    # Insert fact
    dwh_cursor.execute("""
    INSERT INTO Fct_Onderhoud (
        onderhoudnr,
        datum_sk,
        starttijd,
        eindtijd,
        fiets_sk,
        monteur_sk
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (onderhoudnr, datum_sk, starttijd, eindtijd, fiets_sk, monteur_sk))

dwh_conn.commit()
logger.success("Fct_Onderhoud load klaar")

Fct_Inkoop MEES

In [ ]:
logger.info("Ophalen Inkoop uit SDM")

sdm_cursor.execute("""
SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    f.inkoopprijs,
    i.fiets,
    NULL as accessoire
FROM Fiets_Inkoop i
JOIN Fiets_Inkoop_Fiets f ON i.fiets = f.fietsnr

UNION ALL

SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    a.inkoopprijs,
    NULL as fiets,
    i.accessoire
FROM Accessoire_Inkoop i
JOIN Accessoire_Inkoop_Accessoire a ON i.accessoire = a.accessoirenr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT inkoopnr FROM Fct_Inkoop")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None

logger.info("Start ETL Fct_Inkoop")

for row in source_data:
    inkoopnr, datum, aantal, inkoopprijs, fietsnr, accessoirenr = row

    # incremental
    if inkoopnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    accessoire_sk = get_accessoire_sk(accessoirenr)

    # validatie
    if (fiets_sk is None and accessoire_sk is None) or \
       (fiets_sk is not None and accessoire_sk is not None):
        logger.info(f"DEBUG: inkoopnr={inkoopnr}, fietsnr={fietsnr}, accessoire={accessoirenr}, fiets_sk={fiets_sk}, accessoire_sk={accessoire_sk}")
        # logger.warning(f"Fout bij inkoop {inkoopnr}: fiets/accessoire conflict")
        continue

    totaalprijs = aantal * inkoopprijs

    dwh_cursor.execute("""
    INSERT INTO Fct_Inkoop (
        inkoopnr,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (inkoopnr, datum_sk, aantal, totaalprijs,
          inkoopprijs, fiets_sk, accessoire_sk))
    
dwh_conn.commit()
logger.success("Fct_Inkoop load klaar")